In [2]:
import pandas as pd
import numpy as np
import imageio
import h5py
import os, json, sys
import math, random
import shutil

## functions

### umap functions

In [3]:
def scale_to_int_coords (array,  target_min, target_max):
    scaled_array = ((array - array.min()) / (array.max() - array.min())) * (target_max - target_min) + target_min
    int_coords = np.round(scaled_array).astype(int)
    return int_coords

In [4]:
# return a numpy array with columns as "UMAP1_scaleInt", "UMAP2_scaleInt", "tile_x", "tile_y"
def convert_UMAP (arr1_size, UMAP):
    # UMAP is 2D numpy array
    UMAP1_min = np.min(UMAP[:, 0])
    UMAP1_max = np.max(UMAP[:, 0])
    UMAP2_min = np.min(UMAP[:, 1])
    UMAP2_max = np.max(UMAP[:, 1])
    UMAP1_size = UMAP1_max - UMAP1_min
    UMAP2_size = UMAP2_max - UMAP2_min
    arr2_size = int(UMAP2_size/UMAP1_size * arr1_size)

    UMAP_df =pd.DataFrame(np.zeros((UMAP.shape[0], 4)), columns=["UMAP1_scaleInt", "UMAP2_scaleInt", "tile_x", "tile_y"])
    
    UMAP_df["UMAP1_scaleInt"] = scale_to_int_coords (UMAP[:, 0], 0, arr1_size)
    UMAP_df["UMAP2_scaleInt"] = scale_to_int_coords (UMAP[:, 1], 0, arr2_size)
    
    # Compute tile coordinates
    UMAP_df["tile_x"] = UMAP_df["UMAP1_scaleInt"] // tilesize
    UMAP_df["tile_y"] = UMAP_df["UMAP2_scaleInt"] // tilesize
    
    # number of non-empty pixels
    pixel_counts = UMAP_df.groupby(['UMAP1_scaleInt', 'UMAP2_scaleInt']).size()
    
    # number of non-empty tiles
    tile_counts = UMAP_df.groupby(['tile_x', 'tile_y']).size()

    # total number of tiles
    total_tile = np.max(UMAP_df['tile_x']) * np.max(UMAP_df['tile_y'])
    
    # Compute average for non-empty tiles
    avg_points_per_non_empty_tile = tile_counts.mean()

    UMAP_df = UMAP_df.to_numpy()
    
    return UMAP_df

In [5]:
# return umap data loss rate if with arr1_size
def UMAP_loss_rate (arr1_size, UMAP):
    # UMAP is 2D numpy array
    UMAP1_min = np.min(UMAP[:, 0])
    UMAP1_max = np.max(UMAP[:, 0])
    UMAP2_min = np.min(UMAP[:, 1])
    UMAP2_max = np.max(UMAP[:, 1])
    UMAP1_size = UMAP1_max - UMAP1_min
    UMAP2_size = UMAP2_max - UMAP2_min
    arr2_size = int(UMAP2_size/UMAP1_size * arr1_size)

    UMAP_df =pd.DataFrame(np.zeros((UMAP.shape[0], 4)), columns=["UMAP1_scaleInt", "UMAP2_scaleInt", "tile_x", "tile_y"])
    
    UMAP_df["UMAP1_scaleInt"] = scale_to_int_coords (UMAP[:, 0], 0, arr1_size)
    UMAP_df["UMAP2_scaleInt"] = scale_to_int_coords (UMAP[:, 1], 0, arr2_size)
    
    # Compute tile coordinates
    UMAP_df["tile_x"] = UMAP_df["UMAP1_scaleInt"] // tilesize
    UMAP_df["tile_y"] = UMAP_df["UMAP2_scaleInt"] // tilesize
    
    # number of non-empty pixels
    pixel_counts = UMAP_df.groupby(['UMAP1_scaleInt', 'UMAP2_scaleInt']).size()
    
    # number of non-empty tiles
    tile_counts = UMAP_df.groupby(['tile_x', 'tile_y']).size()

    # total number of tiles
    total_tile = np.max(UMAP_df['tile_x']) * np.max(UMAP_df['tile_y'])
    
    # Compute average for non-empty tiles
    avg_points_per_non_empty_tile = tile_counts.mean()

    loss_rate = 1-len(pixel_counts)/UMAP_df.shape[0]
    print(arr1_size, arr2_size, len(pixel_counts), len(tile_counts), total_tile, avg_points_per_non_empty_tile, loss_rate)
    
    return loss_rate

### code dictionary.  save value =0 as background function
return categories_int, cat_list

In [6]:
# data : metadata df
# feature: metadata df column
def findCodes(data, feature): 
    categories = data[feature].to_numpy()  # Replace with actual column name
    print (feature)
    print(categories)
    
    from sklearn.preprocessing import LabelEncoder
    
    # Encode categories as integers starting from 1
    le = LabelEncoder()
    mask = ~pd.isna(categories)
    categories_int = np.zeros_like(categories, dtype=int)
    categories_int[mask] = le.fit_transform(categories[mask]) + 1
    print(len(le.classes_))
    
    int_to_cat_dict = {i + 1: cat for i, cat in enumerate(le.classes_)}
    cat_list = list(int_to_cat_dict.values())
    cat_list.insert(0, "background")
    
    print(json.dumps(cat_list, indent=2))
    print(len(cat_list))

    return categories_int, cat_list

### depth_bit function

In [7]:
# depth_bit
# 16, #65536.0
# 8, #256
def find_depth_bit (cat_list):
    if len(cat_list) <= 256:
        depth_bit =8
    else:
        depth_bit = 16
    return depth_bit

### build base layer function
array_2d_cat:  xi, yi, tilex, tiley, pixel_totalcount, pixel_cats

In [8]:
def determine_maxLevel (h, w):
    max_s = max(h, w)
    max_level = math.ceil(math.log2(max_s)) - math.log2(tilesize)
    return int(max_level)

In [9]:
def build_array_2d_cat (UMAP_df, tilesize):
    #from collections import defaultdict
    #from scipy import sparse

    x = UMAP_df[:,0]
    y = UMAP_df[:,1]
    
    # Determine height and width as the next multiple of tilesize
    height = ((y.max() + tilesize) // tilesize) * tilesize
    width = ((x.max() + tilesize) // tilesize) * tilesize
    print(f"Final shape: h {height} x w {width}")
    
    max_level = determine_maxLevel(height, width)   # not the number of levels (e.g. 8), but the max level number (8-1=7)
    print ("Levels:", max_level +1, "Max Level:", max_level)
    
    UMAP_cat = np.hstack([UMAP_df, categories_int.reshape(-1, 1)]) #UMAP1_int, UMAP2_int, tile_x, tile_y, category_int
    UMAP_cat.shape

    # build UMAP_cat
    # build a pixel_keys, and pixel_counts
    # Step 1: Take first two columns
    keys = UMAP_cat[:, :2]
    
    # Step 2: Use np.unique with return_counts
    unique_keys_first, pixel_totalcount = np.unique(keys, axis=0, return_counts=True)
    unique_keys_first, pixel_totalcount, len(pixel_totalcount)

    # build a pixel_keys, and pixel_cats
    keys = UMAP_cat[:, :2]
    cats = UMAP_cat[:, 4]
    
    # Step 1: lexsort keys to group
    sorted_idx = np.lexsort((keys[:, 0], keys[:, 1]))
    keys_sorted = keys[sorted_idx]
    cats_sorted = cats[sorted_idx]
    
    # Step 2: find unique keys and where they start
    unique_keys_second, start_idx, counts = np.unique(keys_sorted, axis=0, return_index=True, return_counts=True)
    
    assert np.array_equal(unique_keys_first, unique_keys_second)
    
    # Step 3: pick a random row from each group
    np.random.seed(42)  # optional for reproducibility
    random_offsets = np.random.randint(0, counts)
    random_idx = start_idx + random_offsets
    pixel_cats = cats_sorted[random_idx]
    unique_keys_second, pixel_cats, len(pixel_cats)

    # put pixel_x, pixel_y, tile_x, tile_y, pixel_counts, pixel_cats together into a 6 column numpy array
    array_2d_cat = np.concatenate(
        [unique_keys_first, unique_keys_first//tilesize, pixel_totalcount[:, None], pixel_cats[:, None]], axis=1
    )

    return array_2d_cat, max_level

In [10]:
def get_umap_df_dim (UMAP_df):
    x = UMAP_df[:,0]
    y = UMAP_df[:,1]
    
    # Determine height and width as the next multiple of tilesize
    height = ((y.max() + tilesize) // tilesize) * tilesize
    width = ((x.max() + tilesize) // tilesize) * tilesize

    return height, width

### downsample category data function

To downsample a 2D categorical array (where each pixel holds an integer representing a category) by a factor of 2, and assign each block’s value by randomly sampling a non-zero category from the block according to frequency,

In [11]:
def downsample_categorical_by_frequency(arr, pixel_area_equivalent, categories_size, 
                                        density_threshold, block_size=2, background_value=0):
    
    # arr:  pixel_x, pixel_y, tile_x, tile_y, pixel_counts, pixel_cats

    new_xy = arr[:, :2] // block_size
    counts_arr = arr[:, 4]
    cats_arr = arr[:, 5]
    
    unique_keys_first, inverse_idx = np.unique(new_xy, axis=0, return_inverse=True)
    pixel_totalcount = np.bincount(inverse_idx, weights=counts_arr).astype(int)

    # Step 1: lexsort keys to group
    sorted_idx = np.lexsort((new_xy[:, 0], new_xy[:, 1]))
    keys_sorted = new_xy[sorted_idx]
    cats_sorted = cats_arr[sorted_idx]
    
    # Step 2: find unique keys and where they start
    unique_keys_second, start_idx, counts = np.unique(keys_sorted, axis=0, return_index=True, return_counts=True)

    assert np.array_equal(unique_keys_first, unique_keys_second)
    
    # Step 3: pick a random row from each group
    random_offsets = np.random.randint(0, counts)
    random_idx = start_idx + random_offsets
    pixel_cats = cats_sorted[random_idx]

    # Step 4, low density pixel with 50% removal rate
    mask_count = pixel_totalcount/pixel_area_equivalent < density_threshold
    # random 0/1 (True ~50% chance) of same shape
    rand_mask = np.random.rand(len(pixel_totalcount)) < 0.5  
    final_mask = mask_count & rand_mask
    pixel_cats[final_mask] = 0
    
    # Combine into final array
    downsampled = np.concatenate([unique_keys_first, unique_keys_first//tilesize, pixel_totalcount[:, None], pixel_cats[:, None]], axis=1)
    
    return downsampled

### save tile function

In [12]:
from concurrent.futures import ThreadPoolExecutor

def saveAsTiles(array_2d_cat, level,
                tile_size, bitdepth, start_chn, build_dir, target_dir):
    
    # array_2d_cat:  pixel_x, pixel_y, tile_x, tile_y, pixel_counts, pixel_cats
    
    print("building tiles for level", level)

    tiles_keys, inverse_idx = np.unique(
        array_2d_cat[:,2:4], axis=0, return_inverse=True
    )

    def process_tile(k_tile):
        k, (tx, ty) = k_tile
        
        # Get all points in this tile
        rows = np.where(inverse_idx == k)[0]
        
        # Compute local coordinates inside tile
        local_x = array_2d_cat[rows, 0] - tx * tile_size
        local_y = array_2d_cat[rows, 1] - ty * tile_size
        cats = array_2d_cat[rows, 5]
    
        if bitdepth == 16:
            tile = np.zeros((tile_size, tile_size), dtype=np.uint16)
        else:
            tile = np.zeros((tile_size, tile_size), dtype=np.uint8)
    
        tile[local_y, local_x] = cats
    
        rawPNGFile = f"{build_dir}/tile_{level}_{tx*tile_size}_{ty*tile_size}.png"
        imageio.imwrite(rawPNGFile, tile)
    
        dim = f'{tile_size}x{tile_size}'
        finalPNGFile = f'{target_dir}/p{start_chn}-{level}-{ty}-{tx}.png'
        os.system(f'convert {rawPNGFile} -crop {dim} -define png:bit-depth={bitdepth} '
                  f'-depth {bitdepth} -background none -extent {dim} -alpha off {finalPNGFile}')

    # --- Parallel execution ---
    num_cpus = os.cpu_count()
    with ThreadPoolExecutor(max_workers=num_cpus) as executor:  # adjust threads as needed
        executor.map(process_tile, enumerate(tiles_keys))
        

### build metadata function

In [13]:
def write_json(reference_name, viz_label, start_chn, cat_list, tilesize, max_level, offset_x, offset_y, image_scalef):
    # read existing one in
    path = os.path.join(target_dir, "metadata.json")
    if os.path.exists(path):
        with open(path, "r") as f:
            J = json.load(f)
            assert(J["tileSize"] == tilesize)
            assert(J["levels"] == max_level + 1)
            assert(J["offset"] == [float(offset_x), float(offset_y)])
            assert(J["image_scalef"] == float(image_scalef))
            
            for i in range(len(J["phenotypes"]), start_chn + 1):
                J["phenotypes"].append({})  # placeholder
    
            J["phenotypes"][start_chn] = {
                "name": viz_label,
                "type": "category",
                "int_to_category": cat_list
            }
    else:
        height, width = get_umap_df_dim (UMAP_df)
        J={}
        J["reference_name"]= reference_name
        J["count"] = UMAP.shape[0]
        
        J["phenotypes"] = [] 
        while len(J["phenotypes"]) <= start_chn:
            J["phenotypes"].append({})  # placeholder
    
        J["phenotypes"][start_chn] = {
            "name": viz_label,
            "type": "category",
            "int_to_category": cat_list
        }
        J["size"] = [
            int(width/math.pow(2, max_level)),
            int(height/math.pow(2, max_level))
        ]
        J["tileSize"] = tilesize
        J["levels"] = max_level + 1
        J["fileformat"] = "png"
        J["offset"]= [float(offset_x), float(offset_y) ]
        J["image_scalef"] = float(image_scalef)
    print(J)
    with open(os.path.join(target_dir, "metadata.json"), "w") as f:
        json.dump(J, f, indent=4)

## set up section

In [17]:
# only parameter to set on the fly, the rest is configured in pyramid_config.json
# all input files are expected to be in results_dir
results_dir = "Wang2025Nature"
pyramid_file = "pyramid_e95_config.json"

In [18]:
# Path to your config file
config_path = os.path.join(results_dir, pyramid_file)

# Open and load the JSON
with open(config_path, "r") as f:
    config = json.load(f)

reference_name = config["reference_name"]
target_dir = config["target_dir"]
umap_file = config["umap_file"]
metadata_file = config["metadata_file"] # expect first column is index

features = config["features"]
feature_labels = config["feature_labels"]
start_chns = config.get("start_chns", list(range(len(features))))

if not isinstance(features, list):
    raise TypeError("'start_chns' must be a list of indices")
if not isinstance(feature_labels, list):
    raise TypeError("'start_chns' must be a list of indices")    
if not isinstance(start_chns, list):
    raise TypeError("'start_chns' must be a list of indices")
assert(len(features) == len(feature_labels))
assert (all(i < len(features) for i in start_chns))

features = [features[i] for i in start_chns] # filter using start_chns for flexibility
feature_labels = [feature_labels[i] for i in start_chns]

build_dir = 'build'
tilesize = 1024

if os.path.exists(build_dir):
    shutil.rmtree(build_dir, ignore_errors=True)
if not os.path.exists(build_dir):
    os.mkdir(build_dir)

os.makedirs(target_dir,exist_ok=True)

In [19]:
start_chns, features, feature_labels

([0, 1, 2, 3, 4, 5, 6],
 ['cell_type',
  'Class',
  'Subclass',
  'development_stage',
  'tissue',
  'donor_id',
  'sex'],
 ['cell_type',
  'Class',
  'Subclass',
  'development_stage',
  'location',
  'donor_id',
  'sex'])

## Run

### load metadata and umap files

In [20]:
data = pd.read_csv(os.path.join(results_dir, metadata_file), sep ='\t')

In [21]:
# load umap data 
UMAP = np.load(os.path.join(results_dir, umap_file))
UMAP.shape

(232328, 2)

### determine tile dimension

In [22]:
print("arr1_size", "arr2_size", "pixel_counts", "tile_counts", "total_tile", "avg_points_per_non_empty_tile", "loss_rate")
for arr1_size in [2**i for i in range(15, 20)]: #20
    loss_rate = UMAP_loss_rate (arr1_size, UMAP)
    if loss_rate < 0.03:
        break
print(arr1_size)

arr1_size arr2_size pixel_counts tile_counts total_tile avg_points_per_non_empty_tile loss_rate
32768 29000 231525 374 896 621.1978609625669 0.0034563203746427185
32768


### build base UMAP

In [23]:
UMAP_df = convert_UMAP (arr1_size, UMAP)

### determine scale and offset

In [24]:
array = UMAP[:,0]
target_max = arr1_size
target_min = 0
image_scalef = 1/(array.max() - array.min()) * (target_max - target_min)
offset_x = - array.min()/ (array.max() - array.min()) * (target_max - target_min) + target_min
print ("image_scalef:", image_scalef)
print ("offset_x:", offset_x)

array = UMAP[:,1]
target_max = np.max(UMAP_df[:,1])
offset_y = - array.min()/ (array.max() - array.min()) * (target_max - target_min) + target_min
print("offset_y", offset_y)

image_scalef: 879.516
offset_x: 20975.965
offset_y 12244.270861148834


## Build tiles

In [25]:
UMAP_df

array([[26589,  8126,    25,     7],
       [28864, 11472,    28,    11],
       [28671,  9143,    27,     8],
       ...,
       [22788,  5708,    22,     5],
       [17980,  8837,    17,     8],
       [23054,  4916,    22,     4]])

In [26]:
for feature, start_chn, viz_label in zip(features, start_chns, feature_labels):
    # code
    categories_int, cat_list = findCodes(data, feature)
    # depth
    depth_bit = find_depth_bit (cat_list)
    # categorical base layer 
    array_2d_cat, max_level = build_array_2d_cat(UMAP_df, tilesize)

    # tiling
    current = array_2d_cat
    if max_level == 6:
        init_density_threshold = 1.0/(4.0) # 7 levels
    elif max_level == 7:
        init_density_threshold = 1.0/(4.0 * 2.0) # 8 levels
    elif max_level == 8:
        init_density_threshold = 1.0/(4.0 * 4.0) # 9 levels
    elif max_level == 9:
        init_density_threshold = 1.0/(4.0 * 4.0 * 4.0) # 10 levels
    elif max_level == 10:
        init_density_threshold = 1.0/(4.0 * 4.0 * 4.0 * 2.0) # 11 levels
    else:
        print ("for max level", max_level +1, "need to figure out the initial density threshold")
    level = max_level
    
    current_base_area = 1
    block_size = 2
    step = 2.0
    threshold_level = max_level
    
    next_base_area =  current_base_area * block_size * block_size
    next_threshold = init_density_threshold/step
    
    print ("Begin downsampling ...")
    categories_size = len(cat_list) - 1   
    while level > 0:
        result =  downsample_categorical_by_frequency( current, next_base_area, categories_size, 
                                                       next_threshold, block_size=block_size)
        level = level - 1 
        print("Downsampled to level:", level, "count", result.shape)

        if level < threshold_level:
            saveAsTiles(result, level, tilesize, depth_bit, start_chn, build_dir, target_dir)
            print("Saved tiles to disk, level", level)
        
        current = result
        next_base_area = next_base_area * block_size * block_size
        next_threshold = next_threshold/step
    
    ## json
    write_json(reference_name, viz_label, start_chn, cat_list, tilesize, max_level, offset_x, offset_y, image_scalef)
    
    ## build base layer tiles
    if threshold_level == max_level:
        saveAsTiles(array_2d_cat, max_level, tilesize, depth_bit, start_chn, build_dir, target_dir)

cell_type
['intratelencephalic-projecting glutamatergic cortical neuron'
 'glutamatergic neuron'
 'L5 extratelencephalic projecting glutamatergic cortical neuron' ...
 'L2/3 intratelencephalic projecting glutamatergic neuron'
 'pvalb GABAergic cortical interneuron'
 'L4 intratelencephalic projecting glutamatergic neuron']
29
[
  "background",
  "Cajal-Retzius cell",
  "GABAergic neuron",
  "L2/3 intratelencephalic projecting glutamatergic neuron",
  "L4 intratelencephalic projecting glutamatergic neuron",
  "L5 extratelencephalic projecting glutamatergic cortical neuron",
  "L5 intratelencephalic projecting glutamatergic neuron",
  "L6 intratelencephalic projecting glutamatergic neuron",
  "L6b glutamatergic cortical neuron",
  "VIP GABAergic cortical interneuron",
  "astrocyte",
  "brain vascular cell",
  "caudal ganglionic eminence derived cortical interneuron",
  "committed oligodendrocyte precursor",
  "corticothalamic-projecting glutamatergic cortical neuron",
  "forebrain radial 